In [1]:
!pip install pyspark

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Spark Assignment") \
    .getOrCreate()

print("Spark Started Successfully")

Spark Started Successfully


In [3]:
from google.colab import files

uploaded = files.upload()

Saving Sample - Superstore.csv to Sample - Superstore.csv


In [4]:
df = spark.read.csv("Sample - Superstore.csv", header=True, inferSchema=True)

df.show(5)
df.printSchema()

+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [5]:
print("Rows before removing duplicates:", df.count())

df = df.dropDuplicates()

print("Rows after removing duplicates:", df.count())

Rows before removing duplicates: 9994
Rows after removing duplicates: 9994


In [6]:
from pyspark.sql.functions import col, count, when

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



In [7]:
from pyspark.sql.functions import col

filtered_df = df.filter((col("Region") == "West") & (col("Category") == "Technology"))

filtered_df.show(10)

+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+----------+-----------+------+---------------+----------+------------+--------------------+--------+--------+--------+---------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|    Customer Name|    Segment|      Country|         City|     State|Postal Code|Region|     Product ID|  Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|   Profit|
+------+--------------+----------+----------+--------------+-----------+-----------------+-----------+-------------+-------------+----------+-----------+------+---------------+----------+------------+--------------------+--------+--------+--------+---------+
|  2001|CA-2017-166128| 4/11/2017| 4/18/2017|Standard Class|   LW-17215|       Luke Weiss|   Consumer|United States|     Pasadena|California|      91104|  West|TEC-AC-10001767|Technology| Accessories|SanDisk Ultra 64 ...|  

In [8]:
from pyspark.sql.functions import count, sum, avg, min, max

df.select(
    count("*").alias("Total_Records"),
    sum("Sales").alias("Total_Sales"),
    avg("Sales").alias("Average_Sales"),
    min("Sales").alias("Minimum_Sales"),
    max("Sales").alias("Maximum_Sales")
).show()

{"ts": "2026-07-16 17:46:00.792", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[CAST_INVALID_INPUT] The value ' Light Blue\"' of the type \"STRING\" cannot be cast to \"DOUBLE\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018", "context": {"file": "java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)", "line": "", "fragment": "sum", "errorClass": "CAST_INVALID_INPUT"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o243.showString.\n: org.apache.spark.SparkNumberFormatException: [CAST_INVALID_INPUT] The value ' Light Blue\"' of the type \"STRING\" cannot be cast to \"DOUBLE\" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018\n== DataFrame ==\n\"sum\" was called f

NumberFormatException: [CAST_INVALID_INPUT] The value ' Light Blue"' of the type "STRING" cannot be cast to "DOUBLE" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"sum" was called from
java.base/jdk.internal.reflect.NativeMethodAccessorImpl.invoke0(Native Method)


In [9]:
print(df.columns)

['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


In [10]:
df.select("Sales").show(20, False)

+--------+
|Sales   |
+--------+
|19.776  |
|88.8    |
|189.588 |
|204.6   |
|92.94   |
|199.95  |
| Ream"  |
|7.38    |
|16.99   |
|16.68   |
|6.63    |
|1633.188|
|10.776  |
|44.95   |
|29      |
|26.38   |
|17.34   |
|767.952 |
|657.504 |
|30.4    |
+--------+
only showing top 20 rows


In [11]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("multiLine", "true") \
    .option("quote", "\"") \
    .option("escape", "\"") \
    .csv("Sample - Superstore.csv")

df.select("Sales").show(20, False)

+--------+
|Sales   |
+--------+
|261.96  |
|731.94  |
|14.62   |
|957.5775|
|22.368  |
|48.86   |
|7.28    |
|907.152 |
|18.504  |
|114.9   |
|1706.184|
|911.424 |
|15.552  |
|407.976 |
|68.81   |
|2.544   |
|665.88  |
|55.5    |
|8.56    |
|213.48  |
+--------+
only showing top 20 rows


In [12]:
from pyspark.sql.functions import count, sum, avg, min, max

df.select(
    count("*").alias("Total_Records"),
    sum("Sales").alias("Total_Sales"),
    avg("Sales").alias("Average_Sales"),
    min("Sales").alias("Minimum_Sales"),
    max("Sales").alias("Maximum_Sales")
).show()

+-------------+-----------------+-----------------+-------------+-------------+
|Total_Records|      Total_Sales|    Average_Sales|Minimum_Sales|Maximum_Sales|
+-------------+-----------------+-----------------+-------------+-------------+
|         9994|2297200.860299955|229.8580008304938|        0.444|     22638.48|
+-------------+-----------------+-----------------+-------------+-------------+



In [13]:
from pyspark.sql.functions import sum

df.groupBy("Category") \
  .agg(sum("Sales").alias("Total_Sales")) \
  .show()

+---------------+-----------------+
|       Category|      Total_Sales|
+---------------+-----------------+
|Office Supplies|719047.0320000029|
|      Furniture|741999.7952999998|
|     Technology|836154.0329999966|
+---------------+-----------------+



In [14]:
from pyspark.sql.functions import col

df = df.withColumnRenamed("Sales", "Total_Sales")
df = df.withColumn("Total_Sales", col("Total_Sales").cast("double"))

df.printSchema()

root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Total_Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



In [15]:
from pyspark.sql.functions import sum

final_df = (
    df.dropDuplicates()
      .dropna()
      .filter(col("Region") == "West")
      .groupBy("Category")
      .agg(sum("Total_Sales").alias("Region_Total_Sales"))
)

final_df.show()

+---------------+------------------+
|       Category|Region_Total_Sales|
+---------------+------------------+
|Office Supplies|220853.24900000013|
|      Furniture| 252612.7435000002|
|     Technology|251991.83199999985|
+---------------+------------------+

